# 00 - Data Access & STAC Search

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook demonstrates how to search for and access satellite data for the Liwa, Abu Dhabi AOI using:
- **Microsoft Planetary Computer** (Sentinel-2, Landsat)
- **EnMAP** (DLR geoservice)
- **Planet Tanager** (open archive via STAC)
- **Copernicus Data Space**

**Target AOI:** Liwa Oasis, Abu Dhabi, UAE (~23.0°N, 53.8°E)
**Date Range:** 2024-10-01 to 2025-09-30 (growing season)

In [ ]:
# Install required packages & set titiler endpoint fix
import os
os.environ["TITILER_ENDPOINT"] = "https://titiler.xyz"
!pip install -q pystac-client planetary-computer stackstac geopandas rasterio rioxarray folium tqdm


In [ ]:
import os
import json
import geopandas as gpd
from shapely.geometry import box, mapping
import folium

# Define Liwa AOI - approximate bounding box for Liwa Oasis agricultural area
# Coordinates: [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
LIWA_BBOX = [53.5, 22.8, 54.2, 23.3]
LIWA_AOI = box(*LIWA_BBOX)

# Save as GeoJSON for reuse (ensure parent directory exists)
os.makedirs("../data/aoi", exist_ok=True)
aoi_gdf = gpd.GeoDataFrame({'name': ['liwa_aoi']}, geometry=[LIWA_AOI], crs='EPSG:4326')
aoi_gdf.to_file('../data/aoi/liwa_aoi.geojson', driver='GeoJSON')

# Visualize on map using Esri World Imagery basemap (No watermark, No API key required)
esri_tile = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
m = folium.Map(location=[23.05, 53.85], zoom_start=10, tiles=esri_tile, attr='Esri World Imagery')
folium.GeoJson(aoi_gdf.__geo_interface__, name='Liwa AOI').add_to(m)
folium.LayerControl().add_to(m)
m


## 1. Microsoft Planetary Computer - Sentinel-2 Search

In [ ]:
import planetary_computer as pc
from pystac_client import Client
import stackstac
import xarray as xr
import rioxarray  # Explicitly import rioxarray to register .rio accessor

# Connect to Planetary Computer STAC API
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace
)

# Search for Sentinel-2 L2A data over Liwa
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=LIWA_BBOX,
    datetime="2024-10-01/2025-09-30",
    query={"eo:cloud_cover": {"lt": 30}},
    max_items=10
)

items = list(search.items())
print(f"Found {len(items)} Sentinel-2 scenes")

for item in items[:5]:
    print(f"  {item.id}: {item.datetime.date()}, cloud={item.properties.get('eo:cloud_cover')}%")


In [ ]:
# Load sample scenes as stackstac DataArray using bounds_latlon
if items:
    item = items[0]
    print(f"Loading: {item.id}")
    
    # Select bands of interest: B02, B03, B04, B08, B11, B12
    bands = ["B02", "B03", "B04", "B08", "B11", "B12"]
    
    # Note: Use bounds_latlon (degrees EPSG:4326) when epsg is set to target projection (UTM 32640)
    stack = stackstac.stack(
        items[:3],
        assets=bands,
        bounds_latlon=LIWA_BBOX,
        resolution=10,  # 10m resolution
        epsg=32640,     # UTM zone 40N for UAE
        chunksize=2048
    )
    
    print(f"Stack shape: {stack.shape}")  # (time, band, y, x)
    print(f"Bands: {list(stack.band.values)}")
    print(f"CRS: {stack.rio.crs}")
    
    # Compute a quick RGB preview (lazy)
    rgb = stack.sel(band=["B04", "B03", "B02"])
    print("RGB stack ready for computation")


## 2. EnMAP Hyperspectral Data Access

In [ ]:
# EnMAP data access via DLR geoservice
# Note: Requires registration at https://enmap.geoservice.dlr.de

ENMAP_STAC_URL = "https://enmap.geoservice.dlr.de/stac"

try:
    enmap_catalog = Client.open(ENMAP_STAC_URL)
    
    enmap_search = enmap_catalog.search(
        collections=["enmap"],
        bbox=LIWA_BBOX,
        datetime="2024-01-01/2025-12-31",
        max_items=5
    )
    
    enmap_items = list(enmap_search.items())
    print(f"Found {len(enmap_items)} EnMAP scenes")
    for item in enmap_items:
        print(f"  {item.id}: {item.datetime.date()}")
except Exception as e:
    print(f"EnMAP catalog notice: {e}")
    print("Note: DLR EnMAP portal undergoes occasional maintenance. Register at: https://enmap.geoservice.dlr.de")


## 3. Planet Tanager Open Archive (Hyperspectral)

In [ ]:
# Planet Tanager open archive via static STAC collection
import pystac
import requests

TANAGER_COLLECTION_URL = "https://www.planet.com/data/stac/tanager-core-imagery/agriculture/collection.json"

try:
    collection = pystac.Collection.from_file(TANAGER_COLLECTION_URL)
    print(f"Planet Tanager Collection: {collection.id}")
    print(f"Description: {collection.description[:100]}...")
    
    item_links = [link.href for link in collection.links if link.rel == "item"]
    print(f"Found {len(item_links)} scenes in agriculture collection")
    
    if item_links:
        item_url = item_links[0]
        if not item_url.startswith("http"):
            base = TANAGER_COLLECTION_URL.rsplit('/', 1)[0]
            item_url = f"{base}/{item_url}"
        sample_item = pystac.Item.from_file(item_url)
        print(f"  Sample Scene ID: {sample_item.id}")
        print(f"  Available Assets: {list(sample_item.assets.keys())}")
except Exception as e:
    print(f"Planet Tanager search info: {e}")
    print("Check: https://developers.planet.com/docs/data/tanager/")


## 4. Cloud-Optimized GeoTIFF (COG) Streaming with GDAL /vsicurl/

In [ ]:
# Direct COG access without downloading - using GDAL /vsicurl/
# This works with any COG URL (Planetary Computer, AWS, etc.)

import os
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_HTTP_MAX_RETRY"] = "3"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"

from osgeo import gdal
gdal.UseExceptions()

# Example: Sentinel-2 COG from AWS (public bucket)
COG_TEMPLATE = (
    "/vsicurl/https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/"
    "{utm_zone}/{lat_band}/{grid_square}/{year}/{month}/"
    "S2{sat}_{utm_zone}{lat_band}{grid_square}_{date}_0_L2A/"
    "{band}.tif"
)

print("GDAL configured for COG streaming")
print("Use gdalinfo /vsicurl/https://.../B04.tif to inspect")
print("Use gdal_translate -srcwin ... /vsicurl/.../B04.tif window.tif to extract")


## 5. Copernicus Data Space (Alternative Access)

In [ ]:
# Copernicus Data Space requires authentication
# Use sentinelhub-py or openeo for programmatic access

print("Copernicus Data Space Access:")
print("1. Browser: https://dataspace.copernicus.eu/")
print("2. Python: pip install sentinelhub")
print("3. openEO: pip install openeo")


## 6. Summary & Next Steps

In [ ]:
# Data Access Summary for Liwa AOI
import json

summary = {
    "AOI": "Liwa Oasis, Abu Dhabi (EPSG:4326 bbox: [53.5, 22.8, 54.2, 23.3])",
    "Primary_Sources": [
        "Sentinel-2 L2A (Planetary Computer) - 10m, 5-day revisit, 13 bands",
        "Landsat 8/9 (Planetary Computer) - 30m, 16-day revisit, thermal band",
    ],
    "Hyperspectral_Sources": [
        "EnMAP (DLR) - 30m, 205 bands, VNIR-SWIR",
        "Planet Tanager (open archive) - 20m, 205 bands, 400-1700nm",
    ],
    "Auxiliary_Data": [
        "Copernicus DEM 30m - topography, slope, drainage",
        "Global Fields Boundary (GeoParquet) - field parcels",
    ],
    "Recommended_Workflow": [
        "1. Search Sentinel-2 on Planetary Computer for cloud-free scenes",
        "2. Download/stream COGs for selected bands",
        "3. Compute spectral indices",
        "4. Build time series for crop monitoring"
    ]
}

print(json.dumps(summary, indent=2))
